In [1]:
from ase.calculators.plumed import Plumed, restart_from_trajectory
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.bussi import Bussi
from ase.io import read,  write, Trajectory
from ase import units

from mace.calculators import mace_mp

import numpy as np

# Load system
atoms = read("../0_system/init_config.xyz")

# Setup MACE calculator
calc = mace_mp(model='mh-0', head='oc20_usemppbe')

# MD settings
temperature = 700 # K
kT = units.kB*temperature
timestep = 0.5 # fs
taut = 100 # fs
total_time = 100 # ps
nb_steps = int((total_time*1000)//timestep)
interval_info = 100 # steps
interval_traj = 100 # steps

# Write PLUMED input file
with open("plumed.dat", "w") as f:
    f.write(f"""
UNITS LENGTH=A ENERGY=eV

Fe: GROUP ATOMS={','.join(map(str, (np.argwhere(atoms.get_atomic_numbers()==26)+1).flatten().tolist()))}
N: GROUP  ATOMS={','.join(map(str, (np.argwhere(atoms.get_atomic_numbers()==7)+1).flatten().tolist()))}

d_N2: DISTANCE ATOMS={np.argwhere(atoms.get_atomic_numbers()==7)[0,0]+1},{np.argwhere(atoms.get_atomic_numbers()==7)[1,0]+1}
com_N2: COM ATOMS=N
c_N_Fe: COORDINATION GROUPA=N GROUPB=Fe R_0=2.5

UPPER_WALLS ARG=d_N2 AT=2 KAPPA=0.2 EXP=2 EPS=0.1
UPPER_WALLS ARG=com_N2.z AT=10 KAPPA=1

opes: OPES_METAD_EXPLORE ARG=d_N2,c_N_Fe PACE=100 BARRIER=1 TEMP={temperature} STATE_WFILE=STATES STATE_WSTRIDE=1*100

PRINT STRIDE={interval_info} ARG=* FILE=COLVAR
FLUSH STRIDE=100
""" )

# Setup PLUMED
plumed_input = open("plumed.dat", "r").read().splitlines()
plumed_calc = Plumed(calc, plumed_input, timestep * units.fs, atoms, kT)
atoms.calc = plumed_calc

# Setup MD dynamics
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature)
dyn = Bussi(atoms, timestep * units.fs, temperature, taut * units.fs)

# Save energies and temperature
energy_log = []

def log_status(a=atoms, dyn=dyn):
    epot = float(a.get_potential_energy()[0])
    ekin = float(a.get_kinetic_energy())
    etot = epot + ekin
    temp = float(a.get_temperature())
    time_fs = dyn.get_time() / units.fs

    energy_log.append([time_fs, epot, ekin, etot, temp])

dyn.attach(log_status, interval_info)

# Save trajectory
traj = Trajectory("traj.traj", "w", atoms)
dyn.attach(traj, interval_traj)

# Run simulation
dyn.run(nb_steps)

np.savetxt(
    "ENERGY",
    np.asarray(energy_log),
    delimiter=" ",
    header="time_fs Epot_eV Ekin_eV Etot_eV Temp_K",
    fmt="%12.6f"
)

traj = read("traj.traj", index=":")
write("traj.xyz", traj)

/home/lbonati@iit.local/software/miniforge3-mamba/envs/compcatschool/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using Materials Project MACE for MACECalculator with /home/lbonati@iit.local/.cache/mace/macemh0model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


/home/lbonati@iit.local/software/miniforge3-mamba/envs/compcatschool/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/lbonati@iit.local/software/miniforge3-mamba/envs/compcatschool/lib/python3.12/site-packages/mace/calculators/mace.py:199: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
+++ Loading the PLUMED kernel runtime +++


PLUMED: PLUMED is starting
PLUMED: Version: 2.10.0 (git: Unknown) compiled on May 18 2026 at 20:11:18
PLUMED: Please cite these papers when using PLUMED [1][2]
PLUMED: For further information see the PLUMED web page at http://www.plumed.org
PLUMED: Root: /home/lbonati@iit.local/software/miniforge3-mamba/envs/compcatschool/lib/plumed
PLUMED: LibraryPath: /home/lbonati@iit.local/software/miniforge3-mamba/envs/compcatschool/lib/libplumedKernel.so
PLUMED: For installed feature, see /home/lbonati@iit.local/software/miniforge3-mamba/envs/compcatschool/lib/plumed/src/config/config.txt
PLUMED: Molecular dynamics engine: ASE
PLUMED: Precision of reals: 8
PLUMED: Running over 1 node
PLUMED: Number of threads: 1
PLUMED: Cache line size: 512
PLUMED: Number of atoms: 98
PLUMED: File suffix: 
PLUMED: Timestep: 0.000500
PLUMED: KbT: 5.820122
PLUMED: Relevant bibliography:
PLUMED:   [1] The PLUMED consortium, Nat. Methods 16, 670 (2019)
PLUMED:   [2] Tribello, Bonomi, Branduardi, Camilloni, and Bussi,

PlumedError: 
Action "OPES_METAD_EXPLORE" is not known.
An Action named "OPES_METAD_EXPLORE" is available in module "opes".
Please consider installing PLUMED with that module enabled.